# CS336-RAG — test notebook

A step-by-step test of the CS336 course assistant: data → index → four
search strategies → Hit Rate / MRR → a full RAG answer.

We test **retrieval first** because if the search does not find the right
chunks, the LLM cannot answer well no matter how good the prompt is.

Course: DataTalks LLM Zoomcamp 2026. Based on module 1 (RAG), module 2
(vector search), module 4 (evaluation), module 6 (hybrid search + reranking),
module 7 (end-to-end project example).

## What we are testing

The RAG pipeline answers questions about CS336 from:

- **YouTube lecture transcripts** (with timestamps, so answers can link to the exact moment)
- **Code** from the assignment repo (with file paths)

Run this notebook from the project root or the `notebooks/` folder —
the first cell points at the project code so we can reuse `config`,
`minsearch`, `eval` and `rag` instead of copying code here.

In [1]:
# Point at the project root so we can import the project modules
import sys, os, json, time
if not os.path.exists(os.path.join(os.getcwd(), "config.py")):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from collections import Counter
from config import DATA_DIR, EMBED_MODEL, RERANKER_MODEL
from sentence_transformers import SentenceTransformer
from minsearch import Index

## 1. The data

`ingest.py` downloaded 3 lectures (YouTube transcripts) and one GitHub
repo, split everything into ~512-word chunks with a 128-word overlap, and
saved the chunks to `data/documents.json`.

Every chunk has:

- `content` — the text
- `type` — `transcript` or `code`
- `source` — a clickable URL (YouTube link with timestamp, or GitHub file)
- `timestamp` / `video_id` / `file_path` — metadata used for citations

In [2]:
with open(DATA_DIR / "documents.json", encoding="utf-8") as f:
    docs = json.load(f)

print(f"Total chunks: {len(docs)}")
for t, c in Counter(d["type"] for d in docs).items():
    print(f"  {t}: {c}")

print("\nExample transcript chunk:")
t = next(d for d in docs if d["type"] == "transcript")
print(f"  source : {t['source']}")
print(f"  content: {t['content'][:140]}...")

print("\nExample code chunk:")
c = next(d for d in docs if d["type"] == "code")
print(f"  source : {c['source']}")
print(f"  content: {c['content'][:140]}...")

Total chunks: 3740
  transcript: 477
  code: 3263

Example transcript chunk:
  source : https://youtube.com/watch?v=JuoVZkPBiKk&t=4s
  content: Welcome everyone to CS 33336 language models from scratch. Um, this is a teaching staff. I'm Percy, this is Tatsu, Marcel, Herman, and Steve...

Example code chunk:
  source : https://github.com/stanford-cs336/assignment1-basics/blob/main/AGENTS.md
  content: # AI Agent Guidelines for CS336 at Stanford This file provides instructions for AI coding assistants (like ChatGPT, Claude Code, GitHub Copi...


## 2. The search index

`minsearch.Index` builds two things from the chunks (module 1 + module 2):

1. a **TF-IDF matrix** for keyword search — `fit()`
2. a matrix of **embeddings** for semantic search — `fit_embeddings()`

Both are just numpy matrices, so "search" is cosine similarity. No
database — everything lives in memory.

In [3]:
embed_model = SentenceTransformer(EMBED_MODEL)

idx = Index(text_fields=["content"], keyword_fields=["type"])
idx.fit(docs)
idx.fit_embeddings(embed_model.encode([d["content"] for d in docs]))

print(f"Index ready with {len(docs)} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index ready with 3740 chunks


## 3. Keyword search (TF-IDF)

Finds chunks that share the same words as the query. Fast and good for
exact terms (`bpe`, `flops`), but it misses paraphrases — if the lecture
says "byte pair encoding" and we ask about "BPE", keyword search alone
may fail.

In [4]:
def bm25(query, k=10):
    return idx.search(query, num_results=k)

def show_results(title, results, n=5):
    print(title)
    print("-" * 70)
    for i, r in enumerate(results[:n], 1):
        ts = r.get("timestamp")
        ts_str = f"  [t={ts:.0f}s]" if ts is not None else ""
        print(f"{i}. ({r['type']}){ts_str} {r['source']}")
    print()

q = "BPE tokenization"
show_results(f"TF-IDF results for: {q}", bm25(q))

TF-IDF results for: BPE tokenization
----------------------------------------------------------------------
1. (transcript)  [t=4659s] https://youtube.com/watch?v=JuoVZkPBiKk&t=4659s
2. (transcript)  [t=1669s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1669s
3. (transcript)  [t=1701s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1701s
4. (transcript)  [t=3879s] https://youtube.com/watch?v=JuoVZkPBiKk&t=3878s
5. (code) https://github.com/stanford-cs336/assignment1-basics/blob/main/tests/adapters.py



## 4. Vector search (semantic)

Embeds the query and finds the most similar chunks by cosine similarity
(module 2). It matches by *meaning* instead of exact words, so it can
connect "BPE" to "byte pair encoding". Here it surfaces the lecture
segment where tokenization is actually taught.

In [5]:
def vector(query, k=10):
    qv = embed_model.encode(query)
    return idx.vector_search(qv, num_results=k)

show_results(f"Vector results for: {q}", vector(q))

Vector results for: BPE tokenization
----------------------------------------------------------------------
1. (transcript)  [t=4659s] https://youtube.com/watch?v=JuoVZkPBiKk&t=4659s
2. (code) https://github.com/stanford-cs336/assignment1-basics/blob/main/tests/adapters.py
3. (transcript)  [t=1701s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1701s
4. (transcript)  [t=1764s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1764s
5. (transcript)  [t=3910s] https://youtube.com/watch?v=JuoVZkPBiKk&t=3910s



## 5. Hybrid search (RRF)

Combines both rankings with **Reciprocal Rank Fusion** (module 6). Each
chunk gets `1 / (60 + rank)` from each list. A chunk ranked high in either
list scores well, and a chunk ranked high in **both** gets a boost. This
is why hybrid usually beats either method on its own.

In [6]:
def hybrid(query, k=10):
    qv = embed_model.encode(query)
    return idx.hybrid_search(query, qv, k)

show_results(f"Hybrid results for: {q}", hybrid(q))

Hybrid results for: BPE tokenization
----------------------------------------------------------------------
1. (transcript)  [t=4659s] https://youtube.com/watch?v=JuoVZkPBiKk&t=4659s
2. (transcript)  [t=1701s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1701s
3. (code) https://github.com/stanford-cs336/assignment1-basics/blob/main/tests/adapters.py
4. (transcript)  [t=1733s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1733s
5. (transcript)  [t=4380s] https://youtube.com/watch?v=JuoVZkPBiKk&t=4379s



## 6. Reranking

Hybrid returns the top ~10 candidates. A **cross-encoder** then scores
each (query, chunk) *pair together* — much more precise than cosine
similarity alone — and we keep the best 5. Slower, but it puts the most
on-topic chunk at the top.

In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANKER_MODEL)

def rerank(query, hits, k=5):
    scores = reranker.predict([(query, h["content"]) for h in hits])
    for h, s in zip(hits, scores):
        h["_rs"] = float(s)
    return sorted(hits, key=lambda x: x["_rs"], reverse=True)[:k]

show_results(f"Reranked (from hybrid) for: {q}", rerank(q, hybrid(q)))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranked (from hybrid) for: BPE tokenization
----------------------------------------------------------------------
1. (transcript)  [t=1701s] https://youtube.com/watch?v=JuoVZkPBiKk&t=1701s
2. (transcript)  [t=4659s] https://youtube.com/watch?v=JuoVZkPBiKk&t=4659s
3. (code) https://github.com/stanford-cs336/assignment1-basics/blob/main/tests/adapters.py
4. (transcript)  [t=4380s] https://youtube.com/watch?v=JuoVZkPBiKk&t=4379s
5. (transcript)  [t=2015s] https://youtube.com/watch?v=JuoVZkPBiKk&t=2014s



## 7. Evaluating retrieval: Hit Rate and MRR

We test retrieval with 10 questions in `evaluation/eval_questions.json`.
Each question lists the `expected_sources` it should retrieve (which video
or which file). A retrieved chunk is **relevant** if it matches one of
those sources.

Two metrics from the course (module 4):

- **Hit Rate** — fraction of questions where at least one relevant chunk
  appears in the top-k.
- **MRR** (Mean Reciprocal Rank) — `1 / rank` of the *first* relevant
  chunk, averaged. Rewards strategies that rank relevant chunks higher.

We compare four strategies so we can pick the best one for the app.
The metric helpers live in `eval.py`; we reuse them here.

In [8]:
from eval import is_relevant, hit_rate, mrr

with open(DATA_DIR.parent / "evaluation" / "eval_questions.json", encoding="utf-8") as f:
    questions = json.load(f)

def run_eval(strategy):
    relevance_total, latencies = [], []
    for q in questions:
        start = time.time()
        hits = strategy(q["question"])
        latencies.append(time.time() - start)
        relevance_total.append([is_relevant(h, q) for h in hits])
    return hit_rate(relevance_total), mrr(relevance_total), sum(latencies) / len(latencies)

In [9]:
strategies = {
    "TF-IDF": lambda q: idx.search(q, num_results=10),
    "Vector": lambda q: idx.vector_search(embed_model.encode(q), num_results=10),
    "Hybrid": lambda q: idx.hybrid_search(q, embed_model.encode(q), 10),
    "Hybrid + Rerank": lambda q: rerank(q, idx.hybrid_search(q, embed_model.encode(q), 10)),
}

print(f"{'Strategy':<20} {'Hit Rate':<10} {'MRR':<8} {'Latency':<10}")
print("-" * 48)
for name, fn in strategies.items():
    hr, m, lat = run_eval(fn)
    print(f"{name:<20} {hr:<10.2f} {m:<8.2f} {lat:<10.3f}s")

Strategy             Hit Rate   MRR      Latency   
------------------------------------------------


TF-IDF               1.00       0.80     0.021     s


Vector               0.90       0.90     0.024     s


Hybrid               0.90       0.90     0.045     s


Hybrid + Rerank      0.90       0.90     0.639     s


## 8. The full RAG pipeline

`rag.py` chains everything together: (optional) query rewriting → hybrid
search → rerank → build the prompt with citations → call the LLM → return
a cited answer. The answer cites `[1]`, `[2]`, ... so the user can click
through to the exact timestamp or file.

Here we run each step by hand, reusing the index we built above. (In the
app and the CLI, `rag.rag()` wraps all of this into one call.)

This cell needs `OPENCODE_API_KEY` in `.env`. If the key is missing we
just skip it — the retrieval evaluation above is the part that matters most.

In [10]:
from rag import build_prompt, llm, rewrite_query

# Step 1: (optional) rewrite the query for better retrieval
q2 = rewrite_query(q)          # needs the API key; fails fast without it

# Step 2: reuse the index we built above (hybrid + rerank)
hits = rerank(q2, hybrid(q2))

# Step 3: turn hits into a prompt with citations
prompt = build_prompt(q2, hits)

# Step 4: ask the LLM (max_retries=1 so we fail fast instead of waiting)
try:
    answer, _ = llm(prompt, max_retries=1)
    print(f"QUERY: {q}")
    if q2 != q:
        print(f"REWRITTEN: {q2}")
    print("\nANSWER:")
    print(answer)
    print("\nSOURCES:")
    for i, h in enumerate(hits, 1):
        print(f"  [{i}] {h['source']}")
except Exception as e:
    print(f"RAG call skipped ({e}). Add OPENCODE_API_KEY to .env and run again.")

QUERY: BPE tokenization

ANSWER:
BPE tokenization is a byte pair encoding tokenizer that "intuitively breaks the input into frequently occurring chunks" [1]. It is an "effective heuristic that is data-driven" [2]. Conceptually, it is a segmentation of the text into tokens, and through an efficiency lens, tokenization is good because it takes a long raw byte stream and reduces it into a smaller number of tokens [1].

The algorithm starts with a corpus as one long byte sequence: "each byte starts as a token and then we're going to merge successive pairs of adjacent tokens that are occurred the most frequently" [4]. Merges are ordered by order of creation, represented as tuples of bytes `(<token1>, <token2>)` [3]. The vocabulary maps token IDs to token bytes, and special tokens are kept as single tokens and never split [3].

SOURCES:
  [1] https://youtube.com/watch?v=JuoVZkPBiKk&t=1701s
  [2] https://youtube.com/watch?v=JuoVZkPBiKk&t=4659s
  [3] https://github.com/stanford-cs336/assignmen

## Summary — what we learned

- Retrieval happens **before** the LLM; if it is wrong, the answer is wrong.
- TF-IDF finds exact words (good for code), vector finds meaning (good for
  lecture audio transcribed to text), hybrid combines both via RRF.
- Reranking improves the top results at the cost of a few hundred ms.
- Hit Rate / MRR tell us which strategy is best *before* spending money on
  LLM calls.
- `rag.py` ties it all together and returns citations, so users can verify
  answers against the lecture or the code.

### How this maps to the course

| Concept | Course module | In this project |
|--------|---------------|-----------------|
| RAG flow | 1 | `rag.py` |
| Embeddings + vector search | 2 | `minsearch.py` + MiniLM |
| Evaluation (Hit Rate/MRR, judge) | 4 | `eval.py`, `eval_rag.py` |
| Monitoring | 5 | `app.py` logs, `dashboard.py` |
| Hybrid search + reranking | 6 | `minsearch.py` RRF + cross-encoder |
| End-to-end project | 7 | everything wired together